<a href="https://colab.research.google.com/github/callsourav1979-personal/Assignments_HAAI-/blob/main/CV__Sorting_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Install Libraries**

In [2]:
!pip install -q pypdf python-docx
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image
!pip install -q transformers torch accelerate

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


# **Imports & Device configuration**

In [3]:
import torch
import sys
import pypdf
import docx
import pytesseract
import json
import pandas as pd
from pdf2image import convert_from_path
from pathlib import Path
from pypdf import PdfReader
from docx import Document
import re
from difflib import SequenceMatcher


device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:" , device)

print("Python version:" , sys.version)
print("PyTorch version:" , torch.__version__)
print("CUDA available:" , torch.cuda.is_available())
print("PyPdf version:" , pypdf.__version__)
print("python-docx version:" , docx.__version__)
print("PyTesseract version:" , pytesseract.__version__)

if torch.cuda.is_available():
    print("CUDA version:" , torch.version.cuda)
    print("GPU device name:" , torch.cuda.get_device_name(0))
    print("GPU Memory:" , round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),"GB")
else:
  print("Running on CPU")

Using Device: cpu
Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch version: 2.11.0+cpu
CUDA available: False
PyPdf version: 6.16.2
python-docx version: 1.2.0
PyTesseract version: 0.3.13
Running on CPU


# **Model-1 Loading**

In [4]:
from transformers import AutoTokenizer,AutoModelForCausalLM
import torch

model_name1 = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer=AutoTokenizer.from_pretrained(model_name1,trust_remote_code=True)
if device == "cuda":
   model=AutoModelForCausalLM.from_pretrained(model_name1,
                                             torch_dtype=torch.float16,
                                             trust_remote_code=True
                                             )

   model1=model.to("cuda")
else:
   model=AutoModelForCausalLM.from_pretrained(model_name1,
                                               torch_dtype=torch.float32 ,
                                               trust_remote_code=True)
   model1=model.to("cpu")

print("Model loaded Successfully in :" , device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded Successfully in : cpu


# **Define Helper Functions**

In [5]:

def extract_text_from_file(file_path):
    """
    Extract text from PDF or DOCX files.

      For PDF:
          Extracts texts from all pages.

      For DOCX:
          Extracts texts from normal paragraphs and tables.

      Parameters :
           file_path(str): Path to the PDF or DOCX file.
      Returns :
           str: Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    #---------------------------------
    # PDF
    #---------------------------------
    if extension == '.pdf':
        reader = PdfReader(str(path))

        pages = []
        for page in reader.pages:
            text = page.extract_text()
            if text:
              pages.append(text)
        return '\n'.join(pages).strip()
    #---------------------------------------
    # DOCX
    #---------------------------------------
    elif extension == '.docx':
        document = Document(str(path))

        sections = []
        # Extract normal paragraphs
        for paragraph in document.paragraphs:
            text =  paragraph.text.strip()
            if text:
              sections.append(text)

        # Extract tables
        for table in document.tables:
            sections.append("\n[TABLE ]")
            for row in table.rows:
                row_cells = []
                for cell in row.cells:
                    cell_text = cell.text.strip()
                    if cell_text:
                      row_cells.append(cell_text)
                #Combine cells in the same row
                if row_cells:
                   sections.append(" | ".join(row_cells))
            sections.append("[/TABLE]")
        #Combine paragraphs and tables
        extracted_text = "\n".join(sections).strip()

        return extracted_text

    else:
      raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")


In [6]:

def extract_text_from_pdf_ocr(file_path):
    """
    Extract text from scanned/image based PDF files using OCR

     Parameters :
           file_path(str): Path to the PDF file.
      Returns :
           str: OCR Extracted text from the file.
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    extension = path.suffix.lower()
    if extension != '.pdf':
      raise ValueError(f"This function supports PDF files only.")

    # Convert PDF pages into images
    pages = convert_from_path(str(path), dpi=300)

    extracted_pages = []

    # Process each page
    for page_number , page_image in enumerate(pages, start =1):
        print(f"Processing page {page_number}/{len(pages)}")
        #Run OCR
        text = pytesseract.image_to_string(page_image,config="--psm 6")

        #Remove unnessary whitespace
        text = text.strip()

        #Store page seperately
        page_text = (f"\n---PAGE {page_number} ---\n" f"{(text)}")

    extracted_pages.append(page_text)

    # Combine all pages
    final_text = "\n".join(extracted_pages)

    return final_text.strip()

In [7]:

def extract_document_text(file_path):
  """
  Main document-extraction wrapper
  Automatically selects the appropriate extraction method based on the file type
  and available text.

    Parameters:
       file_path (str) : Path to the resume file

    Returns:
       str : Extracted text from the resume
  """
  path = Path(file_path)

  #1. Check whether the file exists
  if not path.exists():
    raise FileNotFoundError(f"File not found: {file_path}")

  #2. Check supported file types
  extension = path.suffix.lower()
  if extension not in ['.pdf' , '.docx']:
    raise ValueError(f"Unsupported file type: {extension}" "Only PDF and DOCX files are supported.")

  #3 Handle PDF
  if extension == '.pdf':
    print(f"\nProcessng PDF: {path.name}")

    #First try normal PDF text extraction
    text = extract_text_from_file(path)

    #Check whether meaningful text was extracted
    if text and len(text.strip()) >= 100 :
      print("Text layer detected. Using standard PDF extraction.")
      return text.strip()
    if len(text.strip()) < 100 :
      print("Little or no text detected . Swtching to OCR ...")
      text = extract_text_from_pdf_ocr(path)
      return text.strip()

  #4 Handle DOCX
  elif extension == '.docx':
    print(f"\nProcessing DOCX: {path.name}")
    text = extract_text_from_file(path)
    return text.strip()

In [8]:

def clean_and_validate_json(response):
  response = response.strip()
  # Remove opening markdown fence
  if response.startswith("```json"):
    response = response[len("```json"):].strip()
  elif response.startswith("```"):
    response = response[len("```"):].strip()

  # Remove closing markdown fence
  if response.endswith("```"):
    response = response[:-3].strip()

  # Validate JSON
  return json.loads(response)

In [9]:

def convert_jd_to_json(jd_text):

  jd_prompt = """
  DOCUMENT TYPE: JOB DESCRIPTION (JD)

  You are a precise recruitment information extraction assistant.

  Your task is to extract information ONLY from the provided JOB DESCRIPTION.
  Do not use outside knowledge. Do not infer, assume, invent, or fabricate information.

  IMPORTANT:
  This is a JOB DESCRIPTION, NOT a candidate CV.

  Return ONLY valid JSON.
  Do not return explanations, comments, markdown, keywords, or any text outside the JSON object.

  Use EXACTLY this JSON structure:

  {
    "job_title": "",
    "skills": [],
    "experience": [],
    "education": [],
    "responsibilities": []
  }

  FIELD DEFINITIONS:

  1. job_title
    Extract the exact job/position title stated in the JD.

  2. skills
    Extract technical skills, technologies, frameworks, platforms, tools,
    methodologies, architectural patterns, and certifications explicitly
    mentioned as required or preferred in the JD.

  Do not invent related technologies that are not explicitly mentioned.

  3. experience
    IMPORTANT: For a JOB DESCRIPTION, "experience" means
    EMPLOYER-STATED EXPERIENCE REQUIREMENTS.

  It does NOT mean candidate employment history.

  Extract experience requirements such as:
  - minimum total years of professional/software engineering experience
  - minimum years of architectural or technical leadership experience
  - required years of experience in a particular area
  - required experience with particular types of systems
  - required experience with methodologies or environments

  Preserve the meaning and wording of the JD as closely as possible.

  For example, if the JD says:
  "Minimum of 8+ years of total software engineering experience"

  then return:

  "experience": [
    "Minimum of 8+ years of total software engineering experience"
  ]

  If the JD says:
  "at least 3+ years acting in a dedicated Architectural or Tech Lead capacity"

  then return:

  "experience": [
    "At least 3+ years acting in a dedicated Architectural or Tech Lead capacity"
  ]

  NEVER convert an experience requirement into a fake employment-history object.

  DO NOT create fields such as:
  "title", "company", "location", "duration", or "description"
  inside the JD experience array.

  4. education
  Extract ONLY education requirements explicitly stated in the JD.

  For example, if the JD says:
  "Bachelor's or Master's degree in Computer Science, Software Engineering,
  or an equivalent technical field."

  return the relevant education requirement using the information actually
  present in the JD.

  DO NOT invent:
  - university names
  - graduation years
  - degree dates
  - fields of study not stated in the JD
  - candidate education details

  5. responsibilities
  Extract responsibilities, duties, activities, and expectations explicitly
  stated in the JD.

  Preserve the meaning of the JD.

  CRITICAL RULES:

  1. Extract information ONLY from the provided JD.
  2. Do NOT use outside knowledge.
  3. Do NOT infer or assume missing information.
  4. Do NOT invent companies, universities, candidates, dates, job histories,
     qualifications, or other information.
  5. Do NOT create candidate employment history from a JD.
  6. For a JD, the "experience" field contains EMPLOYER REQUIREMENTS,
     not candidate work history.
  7. For a JD, experience items must be strings, not employment-history objects.
  8. For a JD, do not create "Company A", "Company B", "University X",
     or similar placeholder/fabricated values.
  9. If information is not available, return [] for list fields and "" for
     the job_title field.
  10. Do not output values such as "Unknown", "Not specified",
     "Not mentioned", or "None".
  11. Do not include personal information that is not relevant to the
     requested extraction.
  12. Do not create a "keywords" field.
  13. Do not add any fields to the JSON structure.
  14. Return ONLY the JSON object..

  NOW EXTRACT THE INFORMATION FROM THIS JOB_DESCRIPTION:
  ----------------------------JOB DESCRIPTION START----------------------------------
  """ + jd_text + """

  ----------------------------JOB DESCRIPTION END----------------------------------

  """

  messages = [
      {"role": "system", "content": "You are a precise recruitment assistant that extracts information from CV"},
      {"role": "user", "content": jd_prompt}
    ]

  text= tokenizer.apply_chat_template(messages , tokenize=False , add_generation_prompt= True)

  inputs = tokenizer(text, return_tensors="pt").to(device)


  with torch.no_grad():
    outputs = model1.generate(**inputs,max_new_tokens=500,do_sample=False)

  jd_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

  return clean_and_validate_json(jd_response)


In [41]:
def convert_cv_to_json(cv_text):

    cv_prompt = """
You are an expert recruitment assistant.

Your task is to analyze the following CV/Resume and convert information explicitly stated in it into a structured JSON format.

CV/RESUME:
""" + cv_text + """

Extract the following information:

1. Candidate Name
2. Job title or professional title, if explicitly stated
3. Skills explicitly mentioned anywhere in the CV
4. Work experience explicitly mentioned in the CV
5. Education/qualifications explicitly mentioned in the CV
6. Job responsibilities, duties, projects, or work activities explicitly mentioned in the CV


Return ONLY valid JSON in exactly this format:

{
  "candidate_name": "",
  "job_title": "",
  "skills": [],
  "experience": [],
  "education": [],
  "responsibilities": []
}


IMPORTANT RULES:

1. Extract information only from the CV. Do not use outside knowledge,
   assumptions, or infer skills that are not explicitly mentioned.

2. Extract skills comprehensively from ALL sections of the CV, including:
   - Technical Skills
   - Skills Summary
   - Work Experience
   - Projects
   - Responsibilities
   - Certifications
   - Tools and Technologies

3. "skills" must contain actual skills, abilities, knowledge, tools,
   technologies, software, programming languages, frameworks, platforms,
   methodologies, or competencies explicitly mentioned in the CV.

4. When a skill statement contains multiple individual technologies,
   extract both the broader skill and explicitly mentioned individual tools.

   Example:
   "Vector Databases (Pinecone, Milvus)"

   Extract:
   "Vector Databases"
   "Pinecone"
   "Milvus"

5. Do NOT infer skills based on the candidate's job title.

   For example, do not assume an AI Engineer knows Python unless Python
   is explicitly mentioned in the CV.

6. "experience" must contain explicitly stated work experience details.

7. "education" must contain explicitly stated educational qualifications,
   degrees, diplomas, certifications, or fields of study.

8. "responsibilities" must contain actual duties, projects, work activities,
   or responsibilities explicitly stated in the CV.

9. Do NOT convert responsibilities into skills unless a specific technology,
   tool, programming language, framework, or competency is explicitly
   mentioned.

10. Do NOT convert education into skills.

11. Do NOT infer experience, skills, or qualifications that are not explicitly
    stated.

12. If information is not present, return an empty list [].

13. NEVER output "None specified", "Not specified", "Not mentioned",
    "Unknown", or similar text. Use [] instead.

14. Do NOT add any fields that are not present in the JSON structure.

15. Do not provide explanations, comments, markdown, or text outside
    the JSON object.

16. Return ONLY valid JSON.

17. Stop generating immediately after the closing }.


OUTPUT LIMITS:

- Do not reproduce the CV verbatim.
- Summarize experience entries concisely.
- Maximum 30 skills.
- Maximum 5 education entries.
- Do not repeat skills.
- Do not repeat responsibilities.
- Do not copy entire paragraphs from the CV.
- Extract only information required by the JSON schema.
"""

    messages = [
        {
            "role": "system",
            "content": "You are a precise recruitment assistant that extracts structured information from CVs."
        },
        {
            "role": "user",
            "content": cv_prompt
        }
    ]

    text = tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(text,return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model1.generate(**inputs,max_new_tokens=1200,do_sample=False)

    cv_response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:],skip_special_tokens=True)

    return clean_and_validate_json(cv_response)

## **Define File Processing Functions**

In [11]:

def read_jd_from_folder(jd_folder):
 jd_folder = Path(jd_folder)


 if not jd_folder.exists():
    raise FileNotFoundError(f"JD Folder not found: {jd_folder}")

 # Find the first PDF or DOCX file
 jd_files = list(jd_folder.glob("JD_*.pdf")) + list(jd_folder.glob("JD_*.docx"))

 if not jd_files:
    raise FileNotFoundError("No JD file found in the folder.")

 jd_file = jd_files[0]

 print(f"JD file found: {jd_file}")

  # Use your existing extraction function
 jd_text = extract_document_text(str(jd_file))

 jd_json = convert_jd_to_json(jd_text)
 #print(json.dumps(jd_json,indent=2))

 return jd_json

In [12]:


def extract_multiple_cvs(cv_folder):
    """
    Extract text from all supported CV files in a folder.

    Supported formats:
      - PDF
      - DOCX

    Parameters:
       cv_folder (str) : Path to the folder containing CVs

    Returns:
       dict: Dictionary containing filename and extracted text
    """

    folder = Path(cv_folder)
    if not folder.exists():
        raise FileNotFoundError(f"CV Folder not found: {cv_folder}")

    if not folder.is_dir():
        raise ValueError(f"Path is not a directory: {cv_folder}")

    # Find all PDF and DOCX files
    cv_files = sorted(
                       [ file
                         for file in folder.iterdir()
                         if file.is_file()
                         and file.name.startswith("CV_")
                         and file.suffix.lower() in [".pdf" , ".docx"]
                        ]
                      )
    if not cv_files:
      raise ValueError(f"No PDF or DOCX files found in :  {cv_folder}")

    cv_data = {}

    for cv_file in cv_files:
      print("="*60)
      print(f"Processing CV: {cv_file.name}")
      print("="*60)

      try:
        text = extract_document_text(cv_file)
        cv_data[cv_file.name] = text
        print(f"Characters Extracted: {len(text)}")
      except Exception as e:
        print(f"Error processing {cv_file.name}: {e}")
        cv_data[cv_file.name] = ""

    return cv_data

In [13]:
def process_all_cvs(cv_folder):

  # Extract all CV files
  all_cvs = extract_multiple_cvs(cv_folder)
  print("Total CVs processed:" , len(all_cvs))

  # Store convereted CV JSON
  all_cv_json ={}

  # Convert each CV to JSON
  for filename , cv_text in all_cvs.items():
    print(f"Processing:  {filename}")

    try:
      cv_json = convert_cv_to_json(cv_text)
      all_cv_json[filename] = cv_json

      print(" Sucessfully converted")

    except Exception as e:
       print(f"Error processing {filename}: {e}")

  print("Total CVs converted: " , len(all_cv_json))
  return all_cv_json

# **Define Matching Functions**

In [47]:

# ---------------------------------------------------------
# 1. TEXT NORMALIZATION
# ---------------------------------------------------------

def normalize(text):
    if not text:
        return ""

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+#./ -]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ---------------------------------------------------------
# 2. FLATTEN CV / JD SKILLS
# ---------------------------------------------------------

def get_skills(data):

    skills = data.get("skills", [])

    expanded_skills = []

    for skill in skills:

        # Add original skill
        expanded_skills.append(skill.strip())

        # Extract items inside brackets
        matches = re.findall(r'\((.*?)\)', skill)

        for match in matches:

            # Split comma-separated skills
            sub_skills = match.split(",")

            for sub_skill in sub_skills:
                expanded_skills.append(sub_skill.strip())

    return expanded_skills


# ---------------------------------------------------------
# 3. DIRECT / FUZZY SKILL MATCHING
# ---------------------------------------------------------
def skill_matches(jd_skills, cv_skills):

    matched_skills = []
    missing_skills = []

    # Normalize CV skills
    cv_skills_lower = [skill.lower().strip() for skill in cv_skills]

    for jd_skill in jd_skills:

        jd_skill_lower = jd_skill.lower().strip()

        found = False

        for cv_skill in cv_skills_lower:

            # Exact or partial match
            if (
                jd_skill_lower == cv_skill
                or jd_skill_lower in cv_skill
                or cv_skill in jd_skill_lower
            ):
                found = True
                break

        if found:
            matched_skills.append(jd_skill)
        else:
            missing_skills.append(jd_skill)

    return matched_skills, missing_skills

# ---------------------------------------------------------
# 4. EXPERIENCE TEXT
# ---------------------------------------------------------

def get_experience_text(data):

    text_parts = []

    # Extract experience details
    experiences = data.get("experience", [])

    if isinstance(experiences, list):

        for exp in experiences:

            if not isinstance(exp, dict):
                continue

            title = exp.get("title", "")
            company = exp.get("company", "")
            start_date = exp.get("start_date", "")
            end_date = exp.get("end_date", "")

            text_parts.append(
                f"{title} {company} {start_date} {end_date}"
            )

    # Extract responsibilities / project activities
    responsibilities = data.get("responsibilities", [])

    if isinstance(responsibilities, list):

        for responsibility in responsibilities:

            if isinstance(responsibility, str):
                text_parts.append(responsibility)

    return " ".join(text_parts)

# ---------------------------------------------------------
# 5. EXPERIENCE MATCHING
# ---------------------------------------------------------

def calculate_experience_score(jd, cv):

    # JD required skills
    jd_skills = get_skills(jd)

    # Candidate's actual work experience
    cv_experience_text = get_experience_text(cv)

    if not jd_skills or not cv_experience_text:
        return 0

    jd_skills_normalized = [
        normalize(skill)
        for skill in jd_skills
    ]

    cv_text = normalize(cv_experience_text)

    # Check whether candidate's work experience
    # contains any of the JD's required skills
    matched_skills = []

    for skill in jd_skills_normalized:
        if skill in cv_text:
            matched_skills.append(skill)

    # No relevant technical/domain experience
    if not matched_skills:
        return 0

    # Relevant experience exists.
    # Score based on percentage of JD skills appearing
    # in the candidate's experience.
    score = (
        len(matched_skills) /
        len(jd_skills_normalized)
    ) * 100

    return min(100, round(score))

# ---------------------------------------------------------
# 6. EDUCATION MATCHING
# ---------------------------------------------------------

def calculate_education_score(jd, cv):

    jd_education = jd.get("education", [])
    cv_education = cv.get("education", [])

    # No education requirement in JD
    if not jd_education:
        return 100

    # Candidate has no education information
    if not cv_education:
        return 0

    # Convert JD education into text
    jd_text = normalize(" ".join(
        item if isinstance(item, str) else str(item)
        for item in jd_education
    ))

    # Convert CV education dictionaries into useful text
    cv_parts = []

    for item in cv_education:

        if isinstance(item, dict):
            cv_parts.append(str(item.get("degree", "")))
            cv_parts.append(str(item.get("field_of_study", "")))
        else:
            cv_parts.append(str(item))

    cv_text = normalize(" ".join(cv_parts))

    # --------------------------------
    # FIELD / SUBJECT MATCHING
    # --------------------------------

    technical_fields = [
        "computer science",
        "software engineering",
        "computer engineering",
        "information technology",
        "information systems",
        "electrical engineering",
        "electronics",
        "engineering"
    ]

    jd_fields = [
        field for field in technical_fields
        if field in jd_text
    ]

    cv_fields = [
        field for field in technical_fields
        if field in cv_text
    ]

    # If JD specifies a technical field,
    # candidate should have a related technical field
    if jd_fields:
        if not cv_fields:
            return 0

        if any(field in cv_fields for field in jd_fields):
            return 100

        return 50

    # --------------------------------
    # DEGREE MATCHING
    # --------------------------------

    if "master" in jd_text or "mba" in jd_text:
        if "master" in cv_text or "mba" in cv_text:
            return 100

    if "bachelor" in jd_text:
        if "bachelor" in cv_text or "master" in cv_text:
            return 100

    return 0
# ---------------------------------------------------------
# 7. OVERALL MATCHING
# ---------------------------------------------------------

def match_candidate(jd, cv):

    jd_skills = get_skills(jd)
    cv_skills = get_skills(cv)

    matched_skills, missing_skills = skill_matches(
        jd_skills,
        cv_skills
    )

    # Skill score
    if jd_skills:

        skill_score = round(
            len(matched_skills) /
            len(jd_skills) *
            100
        )

    else:
        skill_score = 0

    # Experience
    experience_score = calculate_experience_score(
        jd,
        cv
    )

    # Education
    education_score = calculate_education_score(
        jd,
        cv
    )

    # Overall
    overall_score = round(
        skill_score * 0.50 +
        experience_score * 0.30 +
        education_score * 0.20
    )

    # Recommendation
    if overall_score >= 80:
        recommendation = "Strong Match"

    elif overall_score >= 60:
        recommendation = "Good Match"

    elif overall_score >= 40:
        recommendation = "Moderate Match"

    elif overall_score >= 20:
        recommendation = "Weak Match"

    else:
        recommendation = "Poor Match"

    candidate_name = cv.get(
        "candidate_name",
        ""
    )

    return {
        "candidate_name": candidate_name,
        "overall_score": overall_score,
        "skill_match_score": skill_score,
        "experience_match_score": experience_score,
        "education_match_score": education_score,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills,
        "recommendation": recommendation
    }

In [15]:
def match_all_cvs (jd_json,all_cv_json):
  results = []

  for filename , cv_json in all_cv_json.items():
    print(f"Matching CV:  {filename}")

    try:
      result = match_candidate(jd_json,cv_json)
      result["cv_filename"] = filename
      results.append(result)

    except Exception as e:
      print(f"Error matching {filename}: {e}")

  return results

# **Define Ranking Function**

In [16]:
def rank_candidates(results):

  ranked_results = sorted(
      results,
      key=lambda x:
      x.get("overall_score",0)
      , reverse=True
      )

  for rank, results in enumerate(ranked_results, start=1):
    results["rank"] = rank

  return ranked_results

# **Define Result Export Functions**

In [17]:

def create_result_table(ranked_results):
    """
    Creates a Pandas DataFrame from ranked candidate results.

    Parameters:
        ranked_results (list): List of ranked candidate dictionaries.

    Returns:
        pd.DataFrame: Formatted candidate ranking table.
    """

    # Convert ranked results into a DataFrame
    results_df = pd.DataFrame(ranked_results)

    # Select and arrange important columns if they exist
    preferred_columns = [
        "rank",
        "candidate_name",
        "cv_filename",
        "overall_score"
    ]

    available_columns = [
        column for column in preferred_columns
        if column in results_df.columns
    ]

    # Keep preferred columns first
    remaining_columns = [
        column for column in results_df.columns
        if column not in available_columns
    ]

    results_df = results_df[
        available_columns + remaining_columns
    ]

    return results_df

In [18]:
def export_results(results_df, ranked_results, output_folder):
    """
    Exports candidate ranking results to CSV and JSON files.

    Parameters:
        results_df (pd.DataFrame): Candidate ranking table.
        ranked_results (list): Ranked candidate results.
        output_folder (str): Directory where output files will be saved.
    """

    # Convert output path to Path object
    output_path = Path(output_folder)

    # Create output directory if it does not exist
    output_path.mkdir(parents=True, exist_ok=True)

    # Define output file paths
    json_file = output_path / "candidate_ranking.json"
    csv_file = output_path / "candidate_ranking.csv"

    # Save ranked results as JSON
    with open(json_file, "w", encoding="utf-8") as f:
        json.dump(
            ranked_results,
            f,
            indent=2,
            ensure_ascii=False
        )

    # Save result table as CSV
    results_df.to_csv(
        csv_file,
        index=False,
        encoding="utf-8"
    )

    print("Results saved successfully.")
    print(f"JSON file: {json_file}")
    print(f"CSV file: {csv_file}")

# **Main Controller Function**

In [19]:
def run_cv_sorting(jd_folder, cv_folder, output_folder):
    """
    Executes the complete CV sorting workflow.

    Workflow:
        1. Read and convert Job Description to JSON.
        2. Read and convert all CVs to JSON.
        3. Match each CV against the Job Description.
        4. Rank candidates based on matching scores.
        5. Create a result table.
        6. Export results to CSV and JSON.

    Parameters:
        jd_folder (str): Path to the folder containing the Job Description.
        cv_folder (str): Path to the folder containing CV documents.
        output_folder (str): Path where output files will be saved.

    Returns:
        pd.DataFrame: Final ranked candidate result table.
    """

    print("=" * 60)
    print("STARTING CV SORTING PROCESS")
    print("=" * 60)

    # --------------------------------------------------
    # Step 1: Read and process Job Description
    # --------------------------------------------------
    print("\nStep 1: Processing Job Description...")

    jd_json = read_jd_from_folder(jd_folder)

    print("Job Description processed successfully.")

    # --------------------------------------------------
    # Step 2: Read and process all CVs
    # --------------------------------------------------
    print("\nStep 2: Processing CVs...")

    all_cv_json = process_all_cvs(cv_folder)

    print(f"Total CVs processed: {len(all_cv_json)}")

    # --------------------------------------------------
    # Step 3: Match CVs against Job Description
    # --------------------------------------------------
    print("\nStep 3: Matching candidates with Job Description...")

    results = match_all_cvs(jd_json, all_cv_json)

    print(f"Total candidates matched: {len(results)}")

    # --------------------------------------------------
    # Step 4: Rank candidates
    # --------------------------------------------------
    print("\nStep 4: Ranking candidates...")

    ranked_results = rank_candidates(results)

    print("Candidate ranking completed.")

    # --------------------------------------------------
    # Step 5: Create result table
    # --------------------------------------------------
    print("\nStep 5: Creating result table...")

    results_df = create_result_table(ranked_results)

    print("Result table created successfully.")

    # --------------------------------------------------
    # Step 6: Export results
    # --------------------------------------------------
    print("\nStep 6: Exporting results...")

    export_results(
        results_df,
        ranked_results,
        output_folder
    )

    print("\n" + "=" * 60)
    print("CV SORTING PROCESS COMPLETED SUCCESSFULLY")
    print("=" * 60)

    return ranked_results , results_df

# **Execution**

In [ ]:
jd_folder = "/content/jds"
cv_folder = "/content/cvs"
output_folder="/content/output"


ranked_results, results_df = run_cv_sorting(
    jd_folder,
    cv_folder,
    output_folder
)

display(results_df)

In [ ]:
##if __name__ == "__main__":

##    import sys

##    if len(sys.argv) != 4:
##        print("Usage:")
##        print("python cv_sorting.py <jd_folder> <cv_folder> <output_folder>")
##        sys.exit(1)

##    jd_folder = sys.argv[1]
##    cv_folder = sys.argv[2]
##    output_folder = sys.argv[3]

##    run_cv_sorting(
##        jd_folder=jd_folder,
##        cv_folder=cv_folder,
##        output_folder=output_folder
##    )

In [43]:

cv_folder = "/content/cvs"
jd_skills = get_skills(jd_json)

all_cv_json = process_all_cvs(cv_folder)


cv_data = all_cv_json["CV_Alexander_Chen_AI_Engineer.docx"]
cv_skills = get_skills(cv_data)

matched_skills, missing_skills = skill_matches(
    jd_skills,
    cv_skills
)

print("JD SKILLS:", jd_skills)
print("CV SKILLS:", cv_skills)

print("\nMATCHED:", matched_skills)
print("\nMISSING:", missing_skills)

Processing CV: CV_Alexander_Chen_AI_Engineer.docx

Processing DOCX: CV_Alexander_Chen_AI_Engineer.docx
Characters Extracted: 2807
Total CVs processed: 1
Processing:  CV_Alexander_Chen_AI_Engineer.docx
 Sucessfully converted
Total CVs converted:  1
JD SKILLS: ['Python', 'PyTorch', 'TensorFlow', 'Hugging Face Ecosystem', 'vector databases', 'Pinecone', 'Milvus', 'Qdrant', 'Chroma', 'Ray', 'Spark', 'Slurm', 'MLflow', 'Kubeflow', 'Weights & Biases', 'distributed machine learning training architectures', 'compute cluster orchestration']
CV SKILLS: ['Large Language Models (LLMs)', 'LLMs', 'Retrieval-Augmented Generation (RAG) architectures', 'RAG', 'High-throughput MLOps pipelines', 'Prompt Engineering', 'Vector Databases (Pinecone, Milvus)', 'Pinecone', 'Milvus', 'Computer Vision', 'Natural Language Processing (NLP)', 'NLP', 'PyTorch', 'Hugging Face Transformers', 'LangChain', 'LlamaIndex', 'TensorFlow', 'Scikit-Learn', 'vLLM', 'TensorRT', 'AWS (SageMaker, EC2)', 'SageMaker', 'EC2', 'GCP', 

In [44]:
results = match_all_cvs(jd_json, all_cv_json)

for result in results:
    print("\nCandidate:", result["candidate_name"])
    print("Skill Score:", result["skill_match_score"])
    print("Experience Score:", result["experience_match_score"])
    print("Education Score:", result["education_match_score"])
    print("Overall Score:", result["overall_score"])
    print("Recommendation:", result["recommendation"])
    print("Matched Skills:", result["matched_skills"])

Matching CV:  CV_Alexander_Chen_AI_Engineer.docx

Candidate: Alexander Chen
Skill Score: 53
Experience Score: 0
Education Score: 100
Overall Score: 46
Recommendation: Moderate Match
Matched Skills: ['Python', 'PyTorch', 'TensorFlow', 'vector databases', 'Pinecone', 'Milvus', 'Spark', 'MLflow', 'Weights & Biases']


In [50]:
print(jd_json)

{'job_title': 'AI Engineer', 'skills': ['Python', 'PyTorch', 'TensorFlow', 'Hugging Face Ecosystem', 'vector databases', 'Pinecone', 'Milvus', 'Qdrant', 'Chroma', 'Ray', 'Spark', 'Slurm', 'MLflow', 'Kubeflow', 'Weights & Biases', 'distributed machine learning training architectures', 'compute cluster orchestration'], 'experience': ['Production experience training, deploying, or fine-tuning machine learning models in a cloud enterprise environment'], 'education': [], 'responsibilities': ['Design, test, and productionize machine learning algorithms, deep learning models, and complex semantic search frameworks.', 'Build and maintain optimized infrastructure for hosting Large Language Models (LLMs), integrating Retrieval-Augmented Generation (RAG) loops.', 'Develop high-throughput, automated data pre-processing and pipeline systems to feed machine learning model pipelines.', 'Optimize inference runtime parameters to ensure models hit strict latency and memory footprint constraints in produ

In [48]:
cv_data = all_cv_json["CV_Alexander_Chen_AI_Engineer.docx"]

print(get_experience_text(cv_data))

Lead AI Engineer Nexus Intelligent Systems Jan 2024 Present Senior AI / Machine Learning Engineer Cortex Data Labs Mar 2021 Dec 2023 Architected an enterprise RAG system serving 50k+ daily active users, improving response accuracy by 35% and reducing contextual hallucinations using advanced reranking and hybrid search algorithms. Spearheaded the fine-tuning and quantization (AWQ/GPTQ) of Llama-3-70B models for domain-specific medical applications, lowering token latency by 42% and hardware hosting costs by $120k annually. Led a team of 4 ML engineers to build an automated prompt evaluation and adversarial red-teaming pipeline, boosting model reliability against prompt injections by 99.4% Deployed a multi-modal computer vision model for automated defect detection in manufacture lines, accelerating inspection throughput by 250% across 4 production sites. Built and maintained continuous integration pipelines for ML models (MLOps) using Kubeflow and AWS SageMaker, decreasing development-to

In [49]:
cv_data = all_cv_json["CV_Alexander_Chen_AI_Engineer.docx"]

experience_score = calculate_experience_score(
    jd_json,
    cv_data
)

print("Experience Score:", experience_score)

Experience Score: 6


In [53]:
import json

def llm_match_candidate(jd, cv):

    matching_prompt = f"""
You are an expert recruitment and candidate experience evaluation assistant.

Your task is to evaluate how well a candidate's PROFESSIONAL EXPERIENCE
matches a Job Description.

You must perform semantic evaluation.

Do not calculate skill match scores, education scores, overall scores,
or recommendations. Those will be calculated separately using Python.

JOB DESCRIPTION JSON:
{json.dumps(jd, indent=2)}

CANDIDATE CV JSON:
{json.dumps(cv, indent=2)}

Evaluate ONLY the candidate's professional experience relevance.

Consider:

1. Job title relevance
   - Compare the candidate's previous job titles with the target job title.

2. Work experience relevance
   - Compare the candidate's actual work history with the JD experience requirements.

3. Responsibilities and project relevance
   - Compare the candidate's responsibilities, projects, and work activities
     with the responsibilities mentioned in the Job Description.

4. Demonstrated technical/domain experience
   - Consider technologies and domains demonstrated through actual work
     responsibilities and projects.
   - Do not assume experience with a technology merely because it is
     related to another technology.

SCORING GUIDELINES:

90-100:
Candidate has highly relevant professional experience and demonstrates
strong alignment with most of the JD responsibilities and experience requirements.

70-89:
Candidate has substantial relevant professional experience and demonstrates
good alignment with the JD responsibilities, with some gaps.

50-69:
Candidate has partially relevant experience but significant gaps exist.

30-49:
Candidate has limited relevant experience.

0-29:
Candidate has little or no relevant professional experience.

IMPORTANT RULES:

1. Use ONLY evidence explicitly present in the supplied JD JSON and CV JSON.

2. Perform semantic comparison rather than simple keyword matching.

3. Do NOT infer that a candidate has experience with a technology unless
   it is explicitly mentioned in their skills, responsibilities, projects,
   or work experience.

4. Do NOT assume that knowledge of one technology means knowledge of
   another related technology.

   Example:
   Experience with Pinecone and Milvus does NOT automatically mean
   experience with Qdrant or Chroma.

5. Do NOT evaluate education.

6. Do NOT calculate skill_match_score.

7. Do NOT calculate overall_score.

8. Do NOT provide a hiring recommendation.

9. The score should reflect actual demonstrated professional experience,
   not just the candidate's job title.

10. Keep the reasoning concise and evidence-based.

Return ONLY valid JSON in exactly this format:

{{
    "candidate_name": "",
    "experience_match_score": 0,
    "matched_experience_areas": [],
    "experience_gaps": [],
    "reasoning": ""
}}

FIELD DEFINITIONS:

- candidate_name:
  Candidate name from the CV.

- experience_match_score:
  Score from 0 to 100 representing semantic relevance of professional
  experience to the Job Description.

- matched_experience_areas:
  List the relevant areas of professional experience supported by evidence
  in the CV.

- experience_gaps:
  List important JD experience areas that are not demonstrated in the CV.

- reasoning:
  Brief evidence-based explanation of the score.

Return ONLY the JSON object.
Do not return markdown, comments, or explanations outside the JSON.
Stop immediately after the closing }}.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise recruitment assistant specializing in "
                "semantic evaluation of professional experience."
            )
        },
        {
            "role": "user",
            "content": matching_prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model1.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=False
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    print("LLM Experience Matching Response:")
    print(response)

    return clean_and_validate_json(response)

In [54]:
alexander_cv = all_cv_json[
    "CV_Alexander_Chen_AI_Engineer.docx"
]

llm_result = llm_match_candidate(
    jd_json,
    alexander_cv
)

print("\nFINAL PARSED RESULT:")
print(llm_result)

LLM Experience Matching Response:
```json
{
    "candidate_name": "Alexander Chen",
    "experience_match_score": 85,
    "matched_experience_areas": ["Production experience training", "deploying/deploying machine learning models", "architecting enterprise RAG systems"],
    "experience_gaps": ["fine-tuning and quantization of Llama-3-70B models", "automated prompt evaluation and adversarial red-teaming pipeline"],
    "reasoning": "The candidate's experience aligns closely with the JD's responsibilities regarding production experience training, deploying/deploying machine learning models, and architected enterprise RAG systems. However, there are gaps in specific areas like fine-tuning and quantization of Llama-3-70B models and the automated prompt evaluation and adversarial red-teaming pipeline."
}
```

FINAL PARSED RESULT:
{'candidate_name': 'Alexander Chen', 'experience_match_score': 85, 'matched_experience_areas': ['Production experience training', 'deploying/deploying machine lea